# Train a Decision Tree Model with Source Data Manipulation

This notebook loads `synth_data_scored.csv`, allows you to manipulate the source data before training, and then trains a Decision Tree model on `Ja`.

It includes a configurable preprocessing step for `persoon_geslacht_vrouw` so you can make the feature more strongly associated with higher target values.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data_path = Path('synth_data_scored.csv')
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
target_column = 'Ja'
manipulated_feature = 'persoon_geslacht_vrouw'

print('Data shape:', df.shape)
print('Target exists:', target_column in df.columns)
print('Manipulated feature exists:', manipulated_feature in df.columns)
print('
Target summary:')
print(df[target_column].describe())
print('
Feature sample:')
print(df[[manipulated_feature, target_column]].head())

## Analyze current relationship between feature and target

Check how `persoon_geslacht_vrouw` is currently distributed across low and high values of `Ja`.

In [ ]:
df['ja_quantile'] = pd.qcut(df[target_column], q=4, labels=False, duplicates='drop')
grouped = df.groupby('ja_quantile')[manipulated_feature].mean().rename('female_rate')
print('Average', manipulated_feature, 'by target quartile:')
print(grouped)

plt.figure(figsize=(6, 4))
grouped.plot(kind='bar')
plt.title('Average persoon_geslacht_vrouw by Ja quartile')
plt.xlabel('Ja quartile (0 = low, 3 = high)')
plt.ylabel('Mean persoon_geslacht_vrouw')
plt.grid(True, alpha=0.3)
plt.show()

## Manipulate the source feature distribution

This helper updates `persoon_geslacht_vrouw` so higher `Ja` values become more likely to have 1.
Adjust `strength` to control the effect size.

In [ ]:
def adjust_gender_probability(df, target_col, feature_col, strength=0.5, random_state=42):
    df = df.copy()
    rng = np.random.default_rng(random_state)
    sorted_idx = df[target_col].rank(method='first') / len(df)
    base_prob = df[feature_col].mean()
    prob = base_prob + strength * (sorted_idx - 0.5)
    prob = np.clip(prob, 0.01, 0.99)
    df[feature_col + '_adjusted'] = rng.binomial(1, prob)
    return df

strength = 0.9  # increase to make the feature more strongly correlated with higher Ja values
df = adjust_gender_probability(df, target_column, manipulated_feature, strength=strength)
print('New feature column:', manipulated_feature + '_adjusted')
print(df[[manipulated_feature, manipulated_feature + '_adjusted', target_column]].head())

## Check the adjusted feature distribution

Compare the original and adjusted gender feature across target quartiles.

In [ ]:
grouped_adj = df.groupby('ja_quantile')[[manipulated_feature, manipulated_feature + '_adjusted']].mean()
print('Mean original and adjusted feature by target quartile:')
print(grouped_adj)

grouped_adj.plot(kind='bar', figsize=(8, 4))
plt.title('Original vs adjusted persoon_geslacht_vrouw by Ja quartile')
plt.xlabel('Ja quartile')
plt.ylabel('Mean feature value')
plt.grid(True, alpha=0.3)
plt.show()

## Feature selection

Select the input features to use for training. The adjusted gender feature is included here, but you can change the list as needed.

In [ ]:
selected_features = [
    'adres_dagen_op_adres',
    'belemmering_dagen_psychische_problemen',
    'belemmering_dagen_lichamelijke_problematiek',
    'typering_dagen_som',
    'belemmering_dagen_financiele_problemen',
    'ontheffing_dagen_hist_vanwege_uw_medische_omstandigheden',
    'relatie_partner_totaal_dagen_partner',
    'persoonlijke_eigenschappen_dagen_sinds_opvoer',
    'persoonlijke_eigenschappen_dagen_sinds_taaleis',
    manipulated_feature + '_adjusted',
    'afspraak_aantal_woorden',
    'persoon_leeftijd_bij_onderzoek'
]

missing = [c for c in selected_features if c not in df.columns]
if missing:
    raise ValueError(f'Missing selected features in the dataset: {missing}')

print('Using', len(selected_features), 'features:')
for feature in selected_features:
    print('-', feature)

In [ ]:
X = df[selected_features].copy()
y = df[target_column].copy()

print('Feature data shape:', X.shape)
print('Target shape:', y.shape)

display(X.describe(include='all').transpose())

## Train / test split and model training

Train a Decision Tree Regressor on the manipulated dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model = DecisionTreeRegressor(max_depth=6, random_state=42)
model.fit(X_train, y_train)

print('Model trained successfully')
print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('Evaluation on test set:')
print(f'  MAE : {mae:.5f}')
print(f'  RMSE: {rmse:.5f}')
print(f'  R2  : {r2:.5f}')

plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Ja')
plt.ylabel('Predicted Ja')
plt.title('Actual vs Predicted values for Ja')
plt.grid(True, alpha=0.3)
plt.show()

## Feature importances

Inspect which selected features contributed most to the trained model.

In [ ]:
importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df)
plt.figure(figsize=(10, 5))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.title('Decision Tree Feature Importances')
plt.xlabel('Importance')
plt.grid(True, alpha=0.3)
plt.show()

print('Tree structure (first levels):')
print(export_text(model, feature_names=selected_features, max_depth=4))